# Topic 10 그림 코드 모음 — 강의용 Colab 데모

**확률통계 · Topic 10 · Law of Large Numbers, Concentration, Central Limit Theorem**

슬라이드 [T10_slides_v2.md](slides/T10_slides_v2.md) 에 쓴 그림 3개를 만드는 코드를 모았다.
슬라이드는 고정된 PNG지만, 이 노트북은 **강의 중 숫자를 바꿔가며 즉석에서 다시 그려볼 수 있다.**

| 그림 | 슬라이드 | 원본 스크립트 |
|---|---|---|
| ① 네 분포, 같은 실험 — 셋은 종 모양, 하나는 아니다 | 6쪽 `[S]` 출발점을 네 가지로 바꿔 본다 | `figs_src_v2/t10v2_four_dists.py` |
| ② Chebyshev — 항상 옳지만 아주 느슨하다 | 21쪽 `[C]` 항상 옳지만, 아주 느슨하다 | `figs_src_v2/t10v2_chebyshev.py` |
| ③ 표준오차 — 정밀도 2배 = 표본 4배 | 26쪽 `[C]` 표준오차 — 폭이 줄어드는 규칙 | `figs_src_v2/t10v2_sqrt_n.py` |

⚠️ 실제 슬라이드 그림은 Windows 로컬에서 `Malgun Gothic`으로 렌더했다. 이 노트북은 Colab(Linux)이라
나눔고딕을 대신 설치해서 쓴다 — 글꼴만 다르고 배치·색·수치는 슬라이드와 동일하다.

⚠️ ① 은 오프너라 이 그림 안에서는 "중심극한정리"라는 말을 쓰지 않는다. 관찰만 남기고
이름은 PART 4에서 붙인다 — 이 노트북도 그 순서를 그대로 따른다.

**맨 위 설정 셀을 한 번 실행한 뒤, 그림 셀은 순서와 상관없이 원하는 것만 실행하면 된다.**

In [ ]:
# 설정: 한글 글꼴 + 스타일 (맨 처음 한 번만 실행)
# Colab은 Linux라 한글 글꼴이 기본으로 없다. 나눔고딕을 설치해 등록한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

try:
    for f in fm.findSystemFonts(fontpaths=["/usr/share/fonts/truetype/nanum"]):
        fm.fontManager.addfont(f)
    KOREAN_FONT = "NanumGothic"
except Exception:
    KOREAN_FONT = "DejaVu Sans"  # 설치 실패 시 - 한글은 깨지지만 그림은 그려진다
    print("나눔고딕 설치/등록 실패 - DejaVu Sans로 대신한다 (한글이 깨질 수 있음)")

# 슬라이드 테마와 같은 색 (assets/deckstyle.py 와 동일)
C = {
    "accent": "#3b4fd8", "teal": "#0d9488", "orange": "#d97d17",
    "purple": "#8b5cf6", "pink": "#d9457f", "ink": "#0f172a",
    "body": "#4b5768", "muted": "#97a3b6", "line": "#e6eaf1", "soft": "#f7f9fc",
}
CYCLE = [C["accent"], C["teal"], C["orange"], C["purple"], C["pink"], C["muted"]]

mpl.rcParams.update({
    "font.family": KOREAN_FONT,
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.5,
    "ytick.labelsize": 12.5,
    "legend.fontsize": 12.5,
    "legend.frameon": False,
    "lines.linewidth": 2.0,
    "axes.prop_cycle": mpl.cycler(color=CYCLE),
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": C["line"],
    "grid.linewidth": 1.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#d7dce6",
    "figure.dpi": 110,
})

## ① 네 분포, 같은 실험 — 셋은 종 모양, 하나는 아니다

윗줄은 원래 분포, 아랫줄은 **`N`개씩 뽑아 평균 낸 값**을 `REPEATS`번 모은 분포다.
Uniform · Exponential · Bernoulli는 출발점이 전부 다른데 아랫줄이 모두 종 모양이 되고 폭도 좁아진다.
Cauchy만 **좁아지지도, 종 모양이 되지도 않는다** — 기댓값과 분산이 정의되지 않기 때문이다(슬라이드 31).

해 볼 것 — `N` 을 2, 5, 100으로 바꿔 본다. 앞의 셋은 `N`이 커질수록 더 좁아지지만 Cauchy는 그대로다.
출력되는 숫자가 슬라이드 7쪽의 표(평균 0.5003 · 표준편차 0.0529 …)와 같다.

In [ ]:
REPEATS, N = 60_000, 30
rng = np.random.default_rng(20260302)          # 개강일 시드 (전 Topic 공통)

DISTS = [
    ("Uniform(0,1)", lambda size: rng.random(size), (0, 1)),
    ("Exponential(1)", lambda size: rng.exponential(1, size), (0, 6)),
    ("Bernoulli(0.2)", lambda size: (rng.random(size) < 0.2).astype(float), (-0.3, 1.3)),
    ("Cauchy", lambda size: np.tan(np.pi * (rng.random(size) - 0.5)), (-8, 8)),  # 역변환 샘플링 (Topic 6)
]

fig, axes = plt.subplots(2, 4, figsize=(13.4, 4.3))

for j, (name, sampler, xlim) in enumerate(DISTS):
    bad = name == "Cauchy"

    # 윗줄: 원래 분포
    ax = axes[0, j]
    raw = sampler(20_000)
    ax.hist(raw, bins=60, range=xlim, density=True,
            color=C["pink"] if bad else C["muted"], edgecolor="none")
    ax.set_title(name, fontsize=14.5, color=C["ink"])
    ax.set_xlim(xlim)
    ax.set_yticks([])
    ax.grid(False)

    # 아랫줄: N개 평균의 분포
    ax = axes[1, j]
    means = sampler((REPEATS, N)).mean(axis=1)   # 행마다 N개 → 행 평균 REPEATS개
    if bad:
        ax.hist(means, bins=220, range=(-8, 8), density=True,
                color=C["pink"], edgecolor="none")
        ax.set_xlim(-8, 8)
        ax.set_title("안 좁아진다", fontsize=13.5, color=C["pink"])
        print(f"{name:<16} n={N} 평균의 표준편차 {means.std():>10.1f} "
              f"| 중앙값 {np.median(means):+.4f}")
    else:
        lo, hi = means.mean() - 4 * means.std(), means.mean() + 4 * means.std()
        ax.hist(means, bins=60, range=(lo, hi), density=True,
                color=C["accent"], edgecolor="none")
        ax.set_xlim(lo, hi)
        ax.set_title("종 모양", fontsize=13.5, color=C["accent"])
        print(f"{name:<16} n={N} 평균 {means.mean():.4f} 표준편차 {means.std():.4f}")
    ax.set_yticks([])
    ax.grid(False)

axes[0, 0].set_ylabel("원래 분포", fontsize=13.5, color=C["body"])
axes[1, 0].set_ylabel(f"{N}개의 평균", fontsize=13.5, color=C["body"])

fig.text(0.5, -0.03, "출발점이 무엇이든 평균은 종 모양이 된다 - 마지막 하나만 빼고",
         ha="center", fontsize=14, fontweight="bold", color=C["ink"])
fig.subplots_adjust(hspace=0.48, wspace=0.16)
plt.show()

## ② Chebyshev — 항상 옳지만 아주 느슨하다

**왼쪽**: 평균에서 $k\sigma$ 이상 떨어질 확률. 주황선이 Chebyshev 한계 $1/k^2$ 이고,
세 분포의 실제 확률(표본 50만 개)은 **모두 그 아래**에 있다. 세로축이 로그라서 격차가 몇 배인지 보인다.

**오른쪽**: 같은 부등식을 표본평균에 적용한 한계 $\sigma^2/(n\varepsilon^2)$ 와 동전 던지기의 실제 확률.
$n$ 이 커지면 한계가 0으로 간다 — **이 한 줄이 대수의 법칙의 증명**이다(슬라이드 19~21).

해 볼 것 — `eps` 를 0.01로 줄이면 한계가 1에 붙어 있는 구간이 길어진다. 한계가 쓸모 있어지려면
$n$ 이 얼마나 커야 하는지 본다. 출력되는 $k=2$, $k=3$ 값이 슬라이드 22쪽 표와 같다.

In [ ]:
rng = np.random.default_rng(20260302)
N = 500_000
ks = np.arange(1, 5.01, 0.25)

SAMPLES = [
    ("Normal", rng.normal(0, 1, N), C["accent"], "-"),
    ("Uniform", rng.random(N), C["teal"], "--"),
    ("Exponential", rng.exponential(1, N), C["purple"], ":"),
]

fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.45))

# --- 왼쪽: 한계 vs 실제
ax = axes[0]
ax.plot(ks, 1 / ks ** 2, color=C["orange"], lw=3.0, label=r"Chebyshev 한계 $1/k^2$")
for name, s, col, ls in SAMPLES:
    mu, sd = s.mean(), s.std()
    actual = [float((np.abs(s - mu) >= k * sd).mean()) for k in ks]
    ax.plot(ks, actual, ls, color=col, lw=2.0, label=f"실제 - {name}")
    print(f"{name:<12} k=2 실제 {float((np.abs(s-mu) >= 2*sd).mean()):.4f}  "
          f"k=3 실제 {float((np.abs(s-mu) >= 3*sd).mean()):.4f}")
ax.set_yscale("log")
ax.set_ylim(1e-5, 1.5)
ax.set_yticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1])
ax.set_yticklabels(["0.001%", "0.01%", "0.1%", "1%", "10%", "100%"])
ax.minorticks_off()          # 로그 눈금의 10^-5 표기가 한글 폰트에서 깨진다
ax.set_xlabel("$k$  — 평균에서 $k\\sigma$ 이상 떨어질 확률")
ax.set_title("어떤 분포든 주황선 아래에 있다", fontsize=14.5)
ax.legend(loc="lower left", fontsize=11.5)
ax.grid(True, axis="both", color=C["line"])

# --- 오른쪽: 표본평균에 적용
ax = axes[1]
ns = np.arange(10, 2001, 10)
sigma, eps = 0.5, 0.03       # 동전(Bernoulli 0.5)의 표준편차, 허용 오차
bound = np.minimum(sigma ** 2 / (ns * eps ** 2), 1.0)   # 확률이라 1에서 자른다
ax.plot(ns, bound, color=C["orange"], lw=3.0,
        label=r"한계 $\sigma^2/(n\varepsilon^2)$")

sim_ns = np.array([50, 100, 200, 500, 1000, 2000])
sim = [float((np.abs(rng.integers(0, 2, size=(20_000, n)).mean(axis=1) - 0.5) >= eps).mean())
       for n in sim_ns]
for n, s in zip(sim_ns, sim):
    print(f"n={n:>5}  P[|Xbar-0.5| >= {eps}] 실제 {s:.4f}  "
          f"한계 {min(sigma**2/(n*eps**2), 1.0):.4f}")
ax.plot(sim_ns, sim, "o", ms=7, color=C["accent"],
        label=f"실제 (동전, $\\varepsilon = {eps}$)")
ax.set_xlabel("표본 크기 $n$")
ax.set_title("$n$ 이 커지면 한계가 0으로 간다", fontsize=14.5)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="upper right", fontsize=11.5)

fig.text(0.5, -0.13, "왼쪽 - 느슨하지만 항상 옳다   |   오른쪽 - 이 한 줄이 대수의 법칙을 증명한다",
         ha="center", fontsize=13.5, fontweight="bold", color=C["ink"])
fig.subplots_adjust(wspace=0.22)
plt.show()

## ③ 표준오차 — 정밀도 2배 = 표본 4배

**왼쪽**: 동전 던지기 표본평균의 표준편차(표준오차)를 $n$ 별로 시뮬레이션한 값과 이론 $\sigma/\sqrt{n}$.
로그-로그에서 **기울기 $-1/2$ 인 직선**이 된다.

**오른쪽**: 원하는 표준오차를 얻는 데 필요한 표본 $n = (\sigma/\text{SE})^2$.
목표를 절반으로 줄일 때마다 표본이 4배가 된다(슬라이드 27 · Topic 1 질문의 답).

해 볼 것 — `target` 에 0.01을 넣어 본다. 슬라이드 28의 "±0.01 을 원하면 9,604번"은
여기에 95% 계수 $1.96^2$ 을 곱한 값이다.

In [ ]:
rng = np.random.default_rng(20260302)
SIGMA = 0.5   # Bernoulli(0.5)의 표준편차

fig, axes = plt.subplots(1, 2, figsize=(12.4, 3.9))

# --- 왼쪽: 로그-로그 표준오차
ax = axes[0]
ns = np.array([2, 5, 10, 20, 50, 100, 200, 500, 1000])
sim = np.array([rng.integers(0, 2, size=(20_000, n)).mean(axis=1).std() for n in ns])
for n, s in zip(ns, sim):
    print(f"n={n:>5}  시뮬 SE {s:.5f}   이론 {SIGMA/np.sqrt(n):.5f}")

ax.plot(ns, SIGMA / np.sqrt(ns), color=C["accent"], lw=2.6,
        label=r"이론 $\sigma/\sqrt{n}$")
ax.plot(ns, sim, "o", ms=7, color=C["orange"], label="시뮬레이션")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("표본 크기 $n$ (로그)")
ax.set_ylabel("표본평균의 표준편차 (로그)")
ax.set_title("직선이고, 기울기가 정확히 $-1/2$ 다", fontsize=14.5)
ax.set_xticks([2, 10, 100, 1000])
ax.set_xticklabels(["2", "10", "100", "1000"])
ax.set_yticks([0.02, 0.05, 0.1, 0.3])
ax.set_yticklabels(["0.02", "0.05", "0.1", "0.3"])
ax.minorticks_off()          # 로그 눈금의 10^-2 표기가 한글 폰트에서 깨진다
ax.legend(loc="upper right")
ax.grid(True, axis="both", color=C["line"])

slope = np.polyfit(np.log(ns), np.log(sim), 1)[0]
print(f"로그-로그 기울기 {slope:.3f}  (이론 -0.5)")

# --- 오른쪽: 필요한 표본 수
ax = axes[1]
target = np.array([0.05, 0.025, 0.0125])
need = (SIGMA / target) ** 2
print("목표 SE별 필요 표본:", dict(zip(target.tolist(), need.tolist())))

bars = ax.bar([f"±{t:g}" for t in target], need,
              color=[C["teal"], C["accent"], C["orange"]], width=0.55)
for b, v in zip(bars, need):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.12, f"{v:,.0f}개",
            ha="center", fontsize=14, fontweight="bold", color=C["ink"])
ax.set_yscale("log")
ax.set_ylim(50, 5000)
ax.set_yticks([100, 400, 1600])
ax.set_yticklabels(["100", "400", "1600"])
ax.minorticks_off()
ax.set_xlabel("원하는 표준오차")
ax.set_title("표준오차를 1/2로 줄이려면 표본은 4배", fontsize=14.5)

fig.subplots_adjust(wspace=0.30)
plt.show()

---
이 노트북은 채점 대상이 아니다. 슬라이드 그림을 재생성하는 [figs_src_v2/](figs_src_v2/) 스크립트가
원본이며, 슬라이드 PNG를 바꾸려면 그쪽을 고치고 다시 실행해야 한다 — 이 노트북은 강의 중
라이브 데모·질의응답용 사본이다. 표준오차를 직접 재고 "$n \ge 30$" 경험칙을 반증하는 실습은
[lab/T10_lab.ipynb](lab/T10_lab.ipynb) 에 있다.